<a href="https://colab.research.google.com/github/JorgeZorrilla/Crash-GeoNN/blob/main/TrainingModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Meter velocidades
- Meter aceleraciones
- Diferenciar los distintos solidos
- Probar lo del CLAMP_BC_IN_ROLLOUT

## Configuration

In [1]:
# Configuration parameters
SEED_NUMBER = 42
MIN_T = 5
STEP = 2 # To select a smaller number of attributes from the database
LAM_BC = 1e-2
CLAMP_BC_IN_ROLLOUT = True
PATIENCE = 20
MAX_EPOCHS = 200
N_LAYERS= 3
HIDDEN= 128

INPUT_DIR="/content/drive/MyDrive/CrashGeoNN/graphs_bc/"
INPUT_DIR="/content/drive/MyDrive/CrashGeoNN/graphs_iteration_2/"


## Install dependencies

In [2]:
# Colab setup: install PyTorch Geometric wheels matching your Torch/CUDA
import torch, sys, os, platform, subprocess, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# This magic line pulls the right wheels for your torch+cuda combo
torch_ver = torch.__version__.split('+')[0]
cuda_tag = (torch.version.cuda or 'cpu').replace('.', '')
index_url = f"https://data.pyg.org/whl/torch-{torch_ver}%2B{cuda_tag}.html"

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric \
  -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)



Torch: 2.8.0+cu126 | CUDA: 12.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 63.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 135.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 125.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.5 MB/s eta 0:00:00
Device: cuda


Mount drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')  # autoriza y usa rutas como '/content/drive/MyDrive/...'


Mounted at /content/drive


Import dependencies

In [4]:
import os, math, random, numpy as np, time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Dict, List, Tuple
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GraphSAGE
from tqdm.auto import tqdm

Utilities

In [5]:
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try: torch.set_float32_matmul_precision("high")
    except: pass

def worker_init_fn(worker_id):
    seed = torch.initial_seed() % 2**31
    np.random.seed(seed + worker_id); random.seed(seed + worker_id)

def check_sim(steps: List[Data], sid: int, max_print_edges=5):
    assert isinstance(steps, (list, tuple)) and len(steps) >= 1, f"[sim {sid}] bad list"
    N = steps[0].x.shape[0]
    E = steps[0].edge_index.shape[1]
    pos0 = getattr(steps[0], 'pos0', None)
    edge_index0 = steps[0].edge_index
    issues = []
    for t, g in enumerate(steps):
        if not isinstance(g, Data): issues.append(f"step {t} not Data"); continue
        if g.x.dim()!=2 or g.y.dim()!=2: issues.append(f"step {t} x/y dim !=2")
        if g.x.shape[0]!=N or g.y.shape[0]!=N: issues.append(f"step {t} N mismatch")
        if g.edge_index.shape[0]!=2 or g.edge_index.shape[1]!=E: issues.append(f"step {t} ei shape mismatch")
        if not torch.equal(g.edge_index, edge_index0): issues.append(f"step {t} ei differs")
        if int(g.edge_index.max()) >= N: issues.append(f"step {t} ei out of range")
        if not torch.isfinite(g.x).all() or not torch.isfinite(g.y).all(): issues.append(f"step {t} NaN/Inf in x/y")
        if hasattr(g, "edge_attr"):
            if not torch.isfinite(g.edge_attr).all(): issues.append(f"step {t} NaN/Inf in edge_attr")
    ei = edge_index0.t().tolist()
    undirected = all(([j,i] in ei) for i,j in ei[:max_print_edges])
    unique_pairs = set(tuple(sorted(e)) for e in ei)
    dup = (len(unique_pairs) * 2 != len(ei))
    print(f"[sim {sid}] N={N} E={E} undirected? {undirected} duplicates? {dup}")
    if issues: print("  Issues:", "; ".join(issues))

def transform_edge_attr(edge_attr: torch.Tensor, edge_scaler):
    if edge_attr is None or edge_scaler is None:
        return edge_attr
    em, es = edge_scaler
    return (edge_attr - em) / es

def strip_bc_from_graphs(graphs):
    """
    graphs: list[Data] (puede ser lista plana o lista de listas por simulación)
    Modifica cada Data in-place: quita la columna de bc de x y la guarda en bc_mask.
    """
    for g in graphs:
        if hasattr(g, "x") and g.x.shape[1] > 3:  # asumimos última col es bc_flag
            print("Removing extra")
            g.x = g.x[:, :3]  # conservar solo desplazamientos


def drop_features_db(db: List[List[Data]], drop_idx: List[int], drop_from_y: bool=False):
    """
    Elimina atributos (columnas) de x (y opcionalmente de y) en TODA la base de datos.

    Args:
        db: List[List[Data]]  -> base de datos completa
        drop_idx: lista de índices de columnas a eliminar
        drop_from_y: si True, también elimina esas columnas de y
    """
    if not drop_idx:
        return db  # nada que hacer

    drop_idx = sorted(set(drop_idx))

    for sim in db:
        for g in sim:
            # --- X ---
            if hasattr(g, "x") and g.x is not None:
                keep_x = [i for i in range(g.x.size(1)) if i not in drop_idx]
                g.x = g.x[:, keep_x]

            # --- Y (opcional) ---
            if hasattr(g, "y") and g.y is not None:
              keep_y = [i for i in range(g.y.size(1)) if i not in drop_idx]
              g.y = g.y[:, keep_y]


    return db

In [6]:
set_seed(SEED_NUMBER)

Load Database

In [7]:
def load_database(path_pt: str) -> List[List[Data]]:
    print("Loading DB from:", path_pt)
    db = torch.load(path_pt, map_location="cpu", weights_only=False)
    assert isinstance(db, (list, tuple)) and all(isinstance(sim, (list, tuple)) for sim in db)
    for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
    lens = [len(s) for s in db]
    print(f"T-1 per sim (min/mean/max): {min(lens)}/{sum(lens)/len(lens):.1f}/{max(lens)}")
    return db
def load_database_dir(path_pt: str, step = 1) -> List[List[Data]]:
    print("Loading DB from:", path_pt)
    db = []
    graphs = os.listdir(path_pt)
    if graphs:
      print(f"Found {len(graphs)} graphs")
      for i in range(0, len(graphs), step):
        print(f"Loading graph {graphs[i]}")
        path = os.path.join(path_pt, graphs[i])
        db.append(torch.load(path, map_location="cpu", weights_only=False))
      assert isinstance(db, (list, tuple)) and all(isinstance(sim, (list, tuple)) for sim in db)
      for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
      lens = [len(s) for s in db]
      print(f"T-1 per sim (min/mean/max): {min(lens)}/{sum(lens)/len(lens):.1f}/{max(lens)}")
      print(f"Loaded ${len(db)} graphs!")
    return db

def split_simulations(all_sim_ids, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.default_rng(seed); ids = np.array(all_sim_ids); rng.shuffle(ids)
    n = len(ids); n_tr = int(n*train_ratio); n_va = int(n*val_ratio)
    return ids[:n_tr].tolist(), ids[n_tr:n_tr+n_va].tolist(), ids[n_tr+n_va:].tolist()

def build_split_from_db(db: List[List[Data]], sim_ids: List[int], min_time_step: int = 0):
    graphs, sim_static = [], {}
    for sid in sim_ids:
        steps_all = db[sid]
        assert len(steps_all) >= 1, f"Simulation {sid} empty."

        # Si no hay suficientes pasos, saltamos la simulación
        if len(steps_all) <= min_time_step:
            print(f"[WARN] sim {sid} skipped: len(steps)={len(steps_all)} <= min_t={min_time_step}")
            continue

        # Filtrado por timestep
        steps = steps_all[min_time_step:]                        # Data_t(min_time_step) .. Data_t(T-2)
        edge_index = steps[0].edge_index
        pos0 = getattr(steps[0], 'pos0', None)
        simulation_id = getattr(steps[0], 'simulation_id', None)
        bc_mask = getattr(steps[0], 'bc_mask', None)
        rigid_mask = getattr(steps[0], 'rigid_mask', None)
        timestep_index = getattr(steps[0], 't_idx', None)
        # fixed_idx = getattr(steps[0], 'fixed_idx', None)
        edge_attr = getattr(steps[0], 'edge_attr', None)

        # K = len(steps) = (T-1 - min_time_step)
        # Estados efectivos: ΔX_{min_time_step} .. ΔX_T  -> T_eff = K + 1
        T_eff = len(steps) + 1

        # Ground-truth a partir de min_time_step: ΔX_{min_time_step+1 .. T}
        gt_disp_eff = torch.stack([d.y for d in steps], dim=0)  # (T_eff-1, N, N_features)

        # Estado inicial para rollout: ΔX_{min_t} (ojo: sin normalizar)
        dx_init = steps_all[min_time_step].x

        # Añadimos los Data filtrados al conjunto de entrenamiento/val/test
        graphs.extend(steps)

        sim_static[sid] = {
            'simulation_id' : simulation_id,
            'bc_mask' : bc_mask,
            'rigid_mask' : rigid_mask,
            'timestep_index' : timestep_index,
            # 'fixed_idx' : fixed_idx,
            'edge_index': edge_index,
            'edge_attr' : edge_attr,     # OJO: aún sin escalar aquí
            'pos0': pos0,
            'T_eff': T_eff,
            'gt_disp_eff': gt_disp_eff,
            'dx_init': dx_init           # punto de partida del rollout
        }

    return graphs, sim_static

In [8]:
# === Cambia esta ruta a tu .pt (Drive o local) ===
# DB_PATH = "/content/drive/MyDrive/CrashGeoNN/GRAPHS.pt"  # p.ej.: "/content/drive/MyDrive/Crash-GeoNN/GRAPHS.pt"
# simulations = load_database(DB_PATH)
DB_PATH = INPUT_DIR  # p.ej.: "/content/drive/MyDrive/Crash-GeoNN/GRAPHS.pt"
simulations = load_database_dir(DB_PATH, STEP)


Loading DB from: /content/drive/MyDrive/CrashGeoNN/graphs_iteration_2/
Found 93 graphs
Loading graph graph_0.pt
Loading graph graph_0100.pt
Loading graph graph_0102.pt
Loading graph graph_012.pt
Loading graph graph_014.pt
Loading graph graph_016.pt
Loading graph graph_018.pt
Loading graph graph_02.pt
Loading graph graph_022.pt
Loading graph graph_024.pt
Loading graph graph_026.pt
Loading graph graph_028.pt
Loading graph graph_03.pt
Loading graph graph_032.pt
Loading graph graph_034.pt
Loading graph graph_036.pt
Loading graph graph_038.pt
Loading graph graph_04.pt
Loading graph graph_042.pt
Loading graph graph_044.pt
Loading graph graph_046.pt
Loading graph graph_048.pt
Loading graph graph_050.pt
Loading graph graph_052.pt
Loading graph graph_054.pt
Loading graph graph_056.pt
Loading graph graph_058.pt
Loading graph graph_060.pt
Loading graph graph_062.pt
Loading graph graph_064.pt
Loading graph graph_066.pt
Loading graph graph_069.pt
Loading graph graph_071.pt
Loading graph graph_073.p

In [10]:
print(f"Number of features before: {simulations[0][0].x.shape[1]}")
print(f"{simulations[0][0].x.shape}")
print(f"{simulations[0][0].y.shape}")
# simulations = drop_features_db(simulations, drop_idx= [3,4,5,6,7,8,9,10], True)
simulations = drop_features_db(simulations,[6,7,8,9,10], True)
print(f"{simulations[0][0].x.shape}")
print(f"{simulations[0][0].y.shape}")
N_FEATS = simulations[0][0].x.shape[1]
assert simulations[0][0].x.shape[1] == simulations[0][0].y.shape[1]
print(f"Number of features after: {simulations[0][0].x.shape[1]}")


all_ids = list(range(len(simulations)))
train_ids, val_ids, test_ids = split_simulations(all_ids, train_ratio=0.7, val_ratio=0.15, seed=42)

Number of features before: 9
torch.Size([3544, 9])
torch.Size([3544, 9])
torch.Size([3544, 6])
torch.Size([3544, 6])
Number of features after: 6


In [11]:


print("Building splits...")
train_graphs, train_static = build_split_from_db(simulations, train_ids, min_time_step=MIN_T)
val_graphs,   val_static   = build_split_from_db(simulations, val_ids, min_time_step=MIN_T)
test_graphs,  test_static  = build_split_from_db(simulations, test_ids, min_time_step=MIN_T)


Building splits...


## Normalization

In [12]:
def fit_scaler(graphs: List[Data], n_disp_ch: int = 3):
    X = torch.cat([g.x for g in graphs], 0); Y = torch.cat([g.y for g in graphs], 0)
    xm, xs = X.mean(0, keepdim=True), X.std(0, keepdim=True).clamp_min(1e-8)
    ym, ys = Y.mean(0, keepdim=True), Y.std(0, keepdim=True).clamp_min(1e-8)
    return (xm, xs), (ym, ys)

def apply_scaler(graphs: List[Data], x_scaler, y_scaler):
    xm, xs = x_scaler; ym, ys = y_scaler
    for g in graphs:
        g.x = (g.x - xm) / xs
        g.y = (g.y - ym) / ys

def fit_edge_attr_scaler(graphs: List[Data]):
    E_list = [g.edge_attr for g in graphs if hasattr(g, "edge_attr") and g.edge_attr is not None]
    if not E_list: return None
    E = torch.cat(E_list, dim=0)
    em, es = E.mean(0, keepdim=True), E.std(0, keepdim=True).clamp_min(1e-8)
    return (em, es)

def apply_edge_attr_scaler(graphs: List[Data], scaler):
    if scaler is None: return
    em, es = scaler
    for g in graphs:
        if hasattr(g, "edge_attr") and g.edge_attr is not None:
            g.edge_attr = (g.edge_attr - em) / es

In [13]:
print("Fitting scalers on TRAIN...")
x_scaler, y_scaler = fit_scaler(train_graphs)
apply_scaler(train_graphs, x_scaler, y_scaler)
apply_scaler(val_graphs,   x_scaler, y_scaler)
apply_scaler(test_graphs,  x_scaler, y_scaler)

edge_scaler = fit_edge_attr_scaler(train_graphs)
apply_edge_attr_scaler(train_graphs, edge_scaler)
apply_edge_attr_scaler(val_graphs,   edge_scaler)
apply_edge_attr_scaler(test_graphs,  edge_scaler)

Fitting scalers on TRAIN...


Models

In [14]:
from torch_geometric.nn import GINEConv, BatchNorm, LayerNorm, GraphNorm

class ImpactGNN(nn.Module):
    def __init__(self, in_ch=3, hidden=128, out_ch=3, layers=3):
        super().__init__()
        self.gnn = GraphSAGE(in_channels=in_ch, hidden_channels=hidden, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, out_ch))
    def forward(self, x, edge_index, edge_attr=None):
        h = self.gnn(x, edge_index)
        return self.head(h)

class ImpactGNN_Edge(nn.Module):
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        convs, norms = [], []
        for l in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden if l>0 else in_ch, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            convs.append(GINEConv(mlp, edge_dim=edge_attr_dim))
            norms.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_ch))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = x
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, edge_index, edge_attr)
            h = bn(h); h = F.relu(h); h = self.dropout(h)
        return self.head(h)

In [15]:
print("Creating loaders...")
loader_kwargs = dict(batch_size=16, shuffle=True, pin_memory=(device=='cuda'),
                      num_workers=0, worker_init_fn=worker_init_fn, persistent_workers=False)
train_loader = DataLoader(train_graphs, **loader_kwargs)
val_loader   = DataLoader(val_graphs,   **{**loader_kwargs, "shuffle": False})
test_loader  = DataLoader(test_graphs,  **{**loader_kwargs, "shuffle": False})

Creating loaders...


Training

In [16]:
def smooth_edge_penalty(pred, target, edge_index, lam=1e-3):
    src, dst = edge_index
    return lam * ((pred[src] - pred[dst]) - (target[src] - target[dst])).pow(2).mean()

def loss_bc_zero_disp(disp_abs_pred: torch.Tensor, bc_mask: torch.Tensor,
                      lam_bc: float = 1e-3):
    if lam_bc <= 0 or bc_mask is None or bc_mask.sum()==0:
        return disp_abs_pred.new_tensor(0.)
    return lam_bc * (disp_abs_pred[bc_mask]**2).mean()

# TODO: DEFINE THE LOSS OF THE RIGID BODY!

def train_epoch(model, loader, opt, device='cuda', lam_smooth=1e-3, scaler=None, max_grad_norm=1.0):
    model.train(); total, nodes = 0.0, 0
    for g in tqdm(loader, leave=False):
        g = g.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            pred = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))
            loss = F.mse_loss(pred, g.y) + smooth_edge_penalty(pred, g.y, g.edge_index, lam=lam_smooth) + loss_bc_zero_disp(pred, g.bc_mask, LAM_BC)
        if scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
        total += loss.item() * g.num_nodes; nodes += g.num_nodes
    return total / max(nodes, 1)

@torch.no_grad()
def eval_epoch(model, loader, device='cuda', lam_smooth=1e-3):
    model.eval(); total, nodes = 0.0, 0
    for g in tqdm(loader, leave=False):
        g = g.to(device)
        pred = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))
        loss = F.mse_loss(pred, g.y) + smooth_edge_penalty(pred, g.y, g.edge_index, lam=lam_smooth)
        total += loss.item() * g.num_nodes
        nodes += g.num_nodes
    return total / max(nodes, 1)

# ---------- Rollout (+ edge_attr) ----------
@torch.no_grad()
def rollout(model, pos0, edge_index, edge_attr, T_eff, x_scaler, y_scaler,
            dx_init=None, device='cuda', clamp_bc=False, bc_mask=None,
            return_full=True):
    xm, xs = x_scaler; ym, ys = y_scaler

    if pos0 is None:
        N = int(edge_index.max().item()) + 1
        pos0 = torch.zeros((N,3), device=device)
    else:
        pos0 = pos0.to(device)

    edge_index = edge_index.to(device)
    edge_attr  = edge_attr.to(device) if edge_attr is not None else None

    # Estado inicial: ΔX_{min_t}
    if dx_init is None:
        dx_t = torch.zeros((pos0.shape[0], 3), device=device)
    else:
        # Remove BC column if it exists
        if dx_init.shape[1] > 3:
            dx_init = dx_init[:, :3]
        dx_t = dx_init.to(device)


    preds = []
    for _ in range(T_eff-1):
        x_in = (dx_t - xm.to(device)) / xs.to(device)
        y_hat = model(x_in, edge_index, edge_attr)          # normalizado
        y_hat = y_hat * ys.to(device) + ym.to(device)       # desnormalizado -> ΔX_{t+1}
        if clamp_bc and bc_mask is not None:
             y_hat[bc_mask] = 0.0 # Clamp fixed nodes to zero displacement
        preds.append(y_hat)
        dx_t = y_hat                                        # siguiente estado
    return torch.stack(preds, dim=0)


Metrics

In [17]:
@torch.no_grad()
def compute_metrics(pred: torch.Tensor, gt: torch.Tensor) -> Dict[str, float]:
    # Only compare the first 3 dimensions (displacements)
    mae = (pred - gt).abs().mean().item()
    rmse = torch.sqrt(((pred - gt) ** 2).mean()).item()
    ade = (pred - gt).abs().mean(dim=(1,2)).mean().item()
    fde = (pred[-1] - gt[-1]).abs().mean().item()
    return {'MAE': mae, 'RMSE': rmse, 'ADE': ade, 'FDE': fde}

def save_checkpoint(path, model, x_scaler, y_scaler, edge_scaler):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(),
                "x_scaler": x_scaler, "y_scaler": y_scaler, "edge_scaler": edge_scaler}, path)
    print("Saved best checkpoint ->", path)

3D Animation

In [18]:
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML

def unique_undirected_edges(edge_index: torch.Tensor):
    ei = edge_index.detach().cpu().numpy().T
    undirected = set()
    for u, v in ei:
        if u == v: continue
        a, b = (u, v) if u < v else (v, u)
        undirected.add((a, b))
    return np.array(list(undirected), dtype=np.int64)

def make_line_collection(pos_xyz: np.ndarray, edges_uv: np.ndarray, color='k', alpha=0.4, linewidth=0.6):
    segs = np.stack([pos_xyz[edges_uv[:,0]], pos_xyz[edges_uv[:,1]]], axis=1)
    return Line3DCollection(segs, linewidths=linewidth, colors=color, alpha=alpha)

def update_line_collection(lc: Line3DCollection, pos_xyz: np.ndarray, edges_uv: np.ndarray):
    segs = np.stack([pos_xyz[edges_uv[:,0]], pos_xyz[edges_uv[:,1]]], axis=1)
    lc.set_segments(segs)

def animate_simulation(sim_info: dict, model, x_scaler, y_scaler, edge_scaler=None, device='cuda',
                       save_path="/content/rollout.mp4", max_edges=4000, elev=25, azim=40, show_inline=True):
    pos0       = sim_info['pos0']
    edge_index = sim_info['edge_index']
    edge_attr  = transform_edge_attr(sim_info.get('edge_attr', None), edge_scaler)  # 👈 escalar
    T_eff      = sim_info['T_eff']
    gt_disp    = sim_info['gt_disp_eff']   # (T_eff-1,N,3)
    dx_init    = sim_info['dx_init']       # ΔX_{min_t}

    # --- BC mask (opcional) ---
    bc_mask = sim_info.get('bc_mask', None)
    fixed_idx = sim_info.get('fixed_idx', None)
    if bc_mask is None and fixed_idx is not None:
        # construye máscara a partir de índices si es lo que guardas
        N = gt_disp.shape[1]
        bc_mask = torch.zeros(N, dtype=torch.bool)
        bc_mask[fixed_idx] = True

    model.eval()
    with torch.no_grad():
        pred_disp = rollout(model, pos0, edge_index, edge_attr, T_eff, x_scaler, y_scaler,
                            dx_init=dx_init, device=device, clamp_bc=CLAMP_BC_IN_ROLLOUT, bc_mask=bc_mask)

    pos0_np = (pos0 if pos0 is not None else torch.zeros_like(pred_disp[0])).detach().cpu().numpy()
    gt_np   = gt_disp.detach().cpu().numpy()
    pr_np   = pred_disp.detach().cpu().numpy()
    Tm1, N, _ = gt_np.shape

    edges_uv = unique_undirected_edges(edge_index)
    if max_edges is not None and len(edges_uv) > max_edges:
        idx = np.random.RandomState(0).choice(len(edges_uv), size=max_edges, replace=False)
        edges_uv = edges_uv[idx]

    all_gt = pos0_np[None,...] + gt_np
    all_pr = pos0_np[None,...] + pr_np
    xyz_min = np.minimum(all_gt.min(axis=(0,1)), all_pr.min(axis=(0,1)))
    xyz_max = np.maximum(all_gt.max(axis=(0,1)), all_pr.max(axis=(0,1)))
    pad = 0.05 * (xyz_max - xyz_min + 1e-9)
    xyz_min -= pad; xyz_max += pad

    fig = plt.figure(figsize=(12,6))
    ax_gt   = fig.add_subplot(121, projection='3d')
    ax_pr   = fig.add_subplot(122, projection='3d')
    for ax, title in [(ax_gt, "Ground Truth"), (ax_pr, "Prediction")]:
        ax.set_xlim([xyz_min[0], xyz_max[0]]); ax.set_ylim([xyz_min[1], xyz_max[1]]); ax.set_zlim([xyz_min[2], xyz_max[2]])
        ax.view_init(elev=elev, azim=azim); ax.set_title(title)

    pos_gt0 = pos0_np + gt_np[0]; pos_pr0 = pos0_np + pr_np[0]
    lc_gt = make_line_collection(pos_gt0, edges_uv, color='tab:green', alpha=0.6, linewidth=0.7)
    lc_pr = make_line_collection(pos_pr0, edges_uv, color='tab:red',   alpha=0.6, linewidth=0.7)
    ax_gt.add_collection3d(lc_gt); ax_pr.add_collection3d(lc_pr)

    # --- Scatter con o sin BC ---
    use_bc = bc_mask is not None
    if use_bc:
        bc_np   = bc_mask.detach().cpu().numpy().astype(bool)
        free_np = ~bc_np

        # GT
        sc_gt_free = ax_gt.scatter(pos_gt0[free_np,0], pos_gt0[free_np,1], pos_gt0[free_np,2],
                                   s=4, c='tab:green', alpha=0.85, marker='o', label='free')
        sc_gt_fix  = ax_gt.scatter(pos_gt0[bc_np,0],   pos_gt0[bc_np,1],   pos_gt0[bc_np,2],
                                   s=14, c='gold',     alpha=0.95, marker='^', label='BC fixed')

        # Pred
        sc_pr_free = ax_pr.scatter(pos_pr0[free_np,0], pos_pr0[free_np,1], pos_pr0[free_np,2],
                                   s=4, c='tab:red',   alpha=0.85, marker='o', label='free')
        sc_pr_fix  = ax_pr.scatter(pos_pr0[bc_np,0],   pos_pr0[bc_np,1],   pos_pr0[bc_np,2],
                                   s=14, c='dodgerblue', alpha=0.95, marker='^', label='BC fixed')

        # leyenda compacta
        ax_gt.legend(loc='upper left', fontsize=8, frameon=False)
        ax_pr.legend(loc='upper left', fontsize=8, frameon=False)
    else:
        sc_gt = ax_gt.scatter(pos_gt0[:,0], pos_gt0[:,1], pos_gt0[:,2], s=2, c='tab:green', alpha=0.8)
        sc_pr = ax_pr.scatter(pos_pr0[:,0], pos_pr0[:,1], pos_pr0[:,2], s=2, c='tab:red',   alpha=0.8)

    def update(frame):
        pos_gt = pos0_np + gt_np[frame]
        pos_pr = pos0_np + pr_np[frame]

        update_line_collection(lc_gt, pos_gt, edges_uv)
        update_line_collection(lc_pr, pos_pr, edges_uv)

        if use_bc:
            # actualiza offsets 3D para cada grupo
            bc_np   = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np

            sc_gt_free._offsets3d = (pos_gt[free_np,0], pos_gt[free_np,1], pos_gt[free_np,2])
            sc_gt_fix._offsets3d  = (pos_gt[bc_np,0],   pos_gt[bc_np,1],   pos_gt[bc_np,2])
            sc_pr_free._offsets3d = (pos_pr[free_np,0], pos_pr[free_np,1], pos_pr[free_np,2])
            sc_pr_fix._offsets3d  = (pos_pr[bc_np,0],   pos_pr[bc_np,1],   pos_pr[bc_np,2])
            artists = (lc_gt, lc_pr, sc_gt_free, sc_gt_fix, sc_pr_free, sc_pr_fix)
        else:
            sc_gt._offsets3d = (pos_gt[:,0], pos_gt[:,1], pos_gt[:,2])
            sc_pr._offsets3d = (pos_pr[:,0], pos_pr[:,1], pos_pr[:,2])
            artists = (lc_gt, lc_pr, sc_gt, sc_pr)

        ax_gt.set_title(f"Ground Truth – step {frame+1}/{Tm1}")
        ax_pr.set_title(f"Prediction – step {frame+1}/{Tm1}")
        return artists

    ani = animation.FuncAnimation(fig, update, frames=Tm1, interval=120, blit=False)
    try:
        ani.save(save_path, writer=animation.FFMpegWriter(fps=8, bitrate=2000))
        print("Saved animation to:", save_path)
    except Exception as e:
        print("FFMpeg failed:", e)
    plt.close(fig)
    if show_inline:
        display(HTML(ani.to_jshtml()))
    return ani



In [ ]:
import time

start = time.time()
print("Building model...")
edge_dim = train_graphs[0].edge_attr.size(1) if hasattr(train_graphs[0], "edge_attr") and train_graphs[0].edge_attr is not None else 0
assert edge_dim > 0, "edge_attr required for GINEConv; si no tienes, cambia a un modelo sin edge_attr."
model = ImpactGNN_Edge(in_ch=N_FEATS, edge_attr_dim=edge_dim, hidden=HIDDEN, out_ch=N_FEATS, layers=N_LAYERS, dropout=0.1).to(device)
opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

best_val = float('inf'); best_state = None
wait = 0
ckpt_path = "/content/best_impact_gnn.pt"  # cambia a tu Drive si quieres

print(f"Training for up to {MAX_EPOCHS} epochs...")
for epoch in range(1, MAX_EPOCHS+1):
    tr = train_epoch(model, train_loader, opt, device, lam_smooth=1e-3, scaler=scaler, max_grad_norm=1.0)
    vl = eval_epoch(model, val_loader, device, lam_smooth=1e-3)
    print(f"[Epoch {epoch:03d}] train {tr:.6f} | val {vl:.6f}")
    if vl + 1e-6 < best_val:
        best_val = vl; wait = 0
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        save_checkpoint(ckpt_path, model, x_scaler, y_scaler, edge_scaler)
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping (patience reached).")
            break

end = time.time()
print(f"Elapsed time for training: {end - start} seconds.")


Building model...
Training for up to 200 epochs...


/tmp/ipython-input-399376920.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))


  0%|          | 0/392 [00:00<?, ?it/s]

/tmp/ipython-input-667339952.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


  0%|          | 0/86 [00:00<?, ?it/s]

...

[Epoch 032] train 0.024280 | val 0.011872


  0%|          | 0/392 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

[Epoch 033] train 0.023842 | val 0.009546


  0%|          | 0/392 [00:00<?, ?it/s]

## Evaluating results

In [ ]:

if best_state is not None: model.load_state_dict(best_state)

print("Evaluating rollout on TEST sims…")
model.eval(); metrics_all = []
for sid, info in test_static.items():
    pos0 = info['pos0']
    edge_index = info['edge_index']
    edge_attr  = transform_edge_attr(info.get('edge_attr', None), edge_scaler)
    T_eff = info['T_eff']
    gt = info['gt_disp_eff'].to(device)
    dx_init = info['dx_init']
    bc_mask = info['bc_mask']

    pred = rollout(model, pos0, edge_index, edge_attr, T_eff, x_scaler, y_scaler, dx_init=dx_init, device=device, clamp_bc=CLAMP_BC_IN_ROLLOUT, bc_mask=bc_mask)
    m = compute_metrics(pred, gt)
    metrics_all.append(m)
    print(f"[SIM {sid}] MAE={m['MAE']:.6f} RMSE={m['RMSE']:.6f} ADE={m['ADE']:.6f} FDE={m['FDE']:.6f}")


if metrics_all:
    avg = {k: float(np.mean([d[k] for d in metrics_all])) for k in metrics_all[0].keys()}
    print("==== TEST AVERAGE ===="); [print(f"{k}: {v:.6f}") for k,v in avg.items()]



## Creating animation

In [ ]:

# Animación de una simulación de test
print("Creating animation...")
sid = next(iter(test_static.keys()))
animation_name = "rollout_test_" + str(sid) + ".mp4"
_ = animate_simulation(test_static[sid], model, x_scaler, y_scaler, edge_scaler=edge_scaler,
                       device=device, save_path="/content/drive/MyDrive/CrashGeoNN/" + animation_name,
                       max_edges=4000, elev=25, azim=40, show_inline=False)
print("Finished!")


In [ ]:
# Animación de una simulación de train
print("Creating animation...")
sid = next(iter(train_static.keys()))
animation_name = "rollout_train_" + str(sid) + ".mp4"
_ = animate_simulation(train_static[sid], model, x_scaler, y_scaler, edge_scaler=edge_scaler,
                       device=device, save_path="/content/drive/MyDrive/CrashGeoNN/" + animation_name,
                       max_edges=4000, elev=25, azim=40, show_inline=False)
print("Finished!")

## Report

Empezaremos a probar el modelo cada vez añadiendo mas datos para comprobar donde tenemos el error y que features dan lugar a errores asi como ver potenciales mejorias o no:
### Delta_x, Delta_y, Delta_z
Tiempo de entrenamiento: 706 s.

==== TEST AVERAGE ====
MAE: 81.606696
RMSE: 141.483942
ADE: 81.606701
FDE: 148.106764